# Calibration and probability quality (Prompt #10, Steps 26-29, 52-56)

Platt and isotonic calibrators are fitted on each fold's inner slice (after the fitting rows, purged, before the block) and scored on the block. The reliability diagram, the ECE and the Brier / log-loss skill say whether a probability of 0.55 means 55 %. Nothing here chooses a threshold.

**This notebook contains no calculations.** Everything is computed by `xauusd_quant.ml` and `xauusd_quant.research.ml_*` and read back from `results/ml_research/`. Generate those first:

```
xq ml-research --timeframe 5m [--stages grid,trees,...]   # cached walk-forward units
xq ml-report --timeframe 5m                              # tables, ML-H ledger rows, plots
xq ml-compare --timeframe 5m --target mean_reversion      # one target, from the units
xq ml-freeze --timeframe 5m                              # pre-registered freeze rules
xq ml-finalize --timeframe 5m                            # artifacts, streaming, latency
xq ml-final-test --model-spec LGBM_REVERSION_5M_H5_V001   # once, frozen specs only
```

**Prediction research only.** Models output probabilities and expected values; there is no BUY / SELL, threshold, position, stop, PnL or execution anywhere. Walk-forward blocks 2011-2021 are development; the reserved period (2022-) is read only by `ml-final-test`, after the specs are frozen. Skill is measured against the constant training mean (log-loss / MSE skill, positive = better than the baseline) and read block by block.

In [ ]:
import sys
import json
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
from IPython.display import Image, display

from xauusd_quant.ml.config import load_ml_config

cfg = load_ml_config(ROOT / "config" / "ml.yaml")

TIMEFRAME = "5m"       # 5m | 15m - change and re-run

out = cfg.results_path / TIMEFRAME
if not (out / "tables").exists():
    print(f"No report for {TIMEFRAME}. Run:  xq ml-research --timeframe {TIMEFRAME} --report")

def table(name):
    """Load a previously generated table (None if it was not written)."""
    path = out / "tables" / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

def plot(name):
    path = out / "plots" / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f"(no plot {path.name})")

def load_json(*parts):
    path = out.joinpath(*parts)
    return json.loads(path.read_text()) if path.exists() else {}

summary = load_json("summary.json")
{k: v for k, v in summary.items() if not isinstance(v, (dict, list))}

## Reliability diagrams

In [ ]:
plot('reliability')

## Calibration by regime and volatility quartile

In [ ]:
plot('calibration_by_regime')

## Raw vs Platt vs isotonic, block by block

In [ ]:
fm = table('fold_metrics')
fm.filter((pl.col('variant') == 'base') & pl.col('log_loss').is_not_null())\
  .group_by('target', 'horizon', 'family', 'calibration').agg(pl.col('log_loss').mean(), pl.col('ece').mean(), pl.col('brier_skill').mean())\
  .sort('target', 'horizon', 'family', 'calibration')

## Distribution of predicted probabilities

In [ ]:
plot('probability_distribution')

## ROC and PR curves (pooled)

In [ ]:
plot('roc_pr')

## Outcome by prediction decile

In [ ]:
plot('prediction_deciles')
table('deciles')

## Groups: regime, volatility, spread, session

In [ ]:
table('groups').filter(pl.col('family') == 'lightgbm')